In [10]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

In [5]:
#armazenamento da chave da API
GROQ_API_KEY = 'randomkey'
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

In [6]:
#carregamento do documento para que possa ser consumido
loader = PyPDFLoader("manual_avancado_nobreak_fxp2000.pdf")
documents = loader.load()

In [7]:
#geração dos embeddings e armazenamento no banco vetorial
embeddings = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(documents,embeddings)
retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [49]:
#modelo de lingaugem usado na conversa e seus parametros
chat = ChatGroq(
    model="qwen/qwen3.8-27b", 
    temperature=0.5,
    top_p=0.9,
    frequency_penalty=0.7,
    api_key=GROQ_API_KEY
)

D:\Anaconda\Lib\site-packages\pydantic\main.py:250: UserWarning: WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
D:\Anaconda\Lib\site-packages\pydantic\main.py:250: UserWarning: WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [52]:
#configuração do histórico em memória, dicionario store armazena as sessoes ativas
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

#prompt para contextualizar o histórico do chat
contextualize_q_system_prompt = (
    "Dado o historico do chat e a ultima pergunta do usuario" 
    "que deve referenciar o contexto no historico, "
    "formule uma pergunta autonoma que pode ser entandida"
    "sem o historico do chat. NAO responda a pergunta, "
    "apenas formule-a se necessario."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ('system', contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ('human', '{input}'),
])

#criando retriever ciente do histórico
history_aware_retriever = create_history_aware_retriever(
    llm = chat,
    retriever = retriever,
    prompt = contextualize_q_prompt
)

#prompt principal de perguntas e respostas
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """Você é um assistente virtual especializado no atendimento ao cliente no nobreak FXP-2000. 
Responda claramente perguntas técnicas, funcionalidades, garantia, manutenção, atualizações e suporte técnico. 
Se a pergunta for irrelevante, responda educadamente recusando a pergunta.

Use estritamente o contexto do manual abaixo para responder:
{context}"""),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

#construção da chain final recomendada
question_answer_chain = create_stuff_documents_chain(chat, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

#envolvendo tudo com o gerenciador de mensagens
chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key = 'input',
    history_messages_key = 'chat_history',
    output_messages_key = 'answer'
)

D:\Anaconda\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
session_id = 'sessao1'
while True:
    pergunta = input('Voce: ')
    if pergunta.lower() in ['sair','quit','exit']:
        print('Ate logo!')
        break
    resposta = chain_with_history.invoke(
        {'input': pergunta},
        config = {'configurable':{'session_id': session_id}}
    )
    print('Assitente> ', resposta['answer'])